# Day 11 — Leakage-Safe Sequence Dataset for LSTM/GRU

**Project:** IPL Player Performance Analytics and Prediction Using Machine Learning and Deep Learning

## Objective
Build chronological, fixed-length sequences from each player's previous five matches to predict their next-match performance. Today is dataset preparation and validation only; model training begins on Day 12.

Targets:
- Batting: `next_match_runs`
- Bowling: `next_match_wickets` and `next_match_runs_conceded`

**Leakage rule:** a sample's inputs may contain only information available before the target match. Target values and current/next match outcomes must never enter the input sequence.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display

current_dir = Path.cwd().resolve()
candidates = [current_dir, *current_dir.parents]
def is_project_root(p):
    return (
        (p / "data" / "processed" / "batting_next_match_features.csv").is_file()
        or ((p / "data" / "processed").is_dir() and (p / "notebooks").is_dir())
    )
PROJECT_ROOT = next((p for p in candidates if is_project_root(p)), None)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(r"D:\Git\Personal\ipl-player-performance")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DAY11_DIR = PROCESSED_DIR / "day11_sequences"
REPORT_DIR = PROJECT_ROOT / "reports" / "day11"
DOCS_DIR = PROJECT_ROOT / "docs"
for folder in [DAY11_DIR, REPORT_DIR, DOCS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

BATTING_PATH = PROCESSED_DIR / "batting_next_match_features.csv"
BOWLING_PATH = PROCESSED_DIR / "bowling_next_match_features.csv"

print("Project root:", PROJECT_ROOT)
for path in [BATTING_PATH, BOWLING_PATH]:
    print(("FOUND  " if path.is_file() else "MISSING"), path)

Project root: D:\Git\Personal\ipl-player-performance
FOUND   D:\Git\Personal\ipl-player-performance\data\processed\batting_next_match_features.csv
FOUND   D:\Git\Personal\ipl-player-performance\data\processed\bowling_next_match_features.csv


## 1. Load data and identify historical input features

Only numeric `prior_...` columns are candidates, matching the project's previous feature-eligibility convention. A feature manifest is saved so Day 12 can use exactly the same feature order.

In [2]:
def load_table(path, required_cols):
    if not path.is_file():
        raise FileNotFoundError(f"Missing {path}. Complete Day 4 feature engineering first.")
    df = pd.read_csv(path)
    missing = set(required_cols) - set(df.columns)
    if missing:
        raise ValueError(f"{path.name} is missing required columns: {sorted(missing)}")
    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df["next_match_date"] = pd.to_datetime(df["next_match_date"], errors="coerce")
    return df

batting = load_table(
    BATTING_PATH,
    {"player_name", "match_id", "date", "next_match_date", "next_match_runs"}
)
bowling = load_table(
    BOWLING_PATH,
    {"player_name", "match_id", "date", "next_match_date",
     "next_match_wickets", "next_match_runs_conceded"}
)

IDENTIFIER_COLS = {
    "player_name", "player_id", "match_id", "next_match_id",
    "date", "next_match_date", "source_file"
}
TARGET_COLS = {
    "next_match_runs", "next_match_wickets", "next_match_runs_conceded"
}
CURRENT_OUTCOME_COLS = {
    "runs", "balls_faced", "fours", "sixes", "strike_rate", "batting_position",
    "dismissed", "wickets", "runs_conceded", "legal_balls", "overs", "economy",
    "maidens", "dots", "wides", "noballs"
}

def candidate_features(df):
    return [
        c for c in df.columns
        if c.startswith("prior_")
        and pd.api.types.is_numeric_dtype(df[c])
        and c not in IDENTIFIER_COLS
        and c not in TARGET_COLS
        and c not in CURRENT_OUTCOME_COLS
    ]

bat_features = candidate_features(batting)
bowl_features = candidate_features(bowling)
print("Batting candidate features:", bat_features)
print("Bowling candidate features:", bowl_features)
if not bat_features:
    raise ValueError("No numeric prior_ batting features found; inspect Day 4 output schema.")
if not bowl_features:
    raise ValueError("No numeric prior_ bowling features found; inspect Day 4 output schema.")

Batting candidate features: ['prior_career_batting_runs', 'prior_last5_mean_batting_runs', 'prior_last5_count_batting_runs', 'prior_career_batting_balls_faced', 'prior_last5_mean_batting_balls_faced', 'prior_last5_count_batting_balls_faced', 'prior_career_batting_fours', 'prior_last5_mean_batting_fours', 'prior_last5_count_batting_fours', 'prior_career_batting_sixes', 'prior_last5_mean_batting_sixes', 'prior_last5_count_batting_sixes', 'prior_career_batting_strike_rate', 'prior_last5_mean_batting_strike_rate', 'prior_last5_count_batting_strike_rate']
Bowling candidate features: ['prior_career_bowling_wickets', 'prior_last5_mean_bowling_wickets', 'prior_last5_count_bowling_wickets', 'prior_career_bowling_runs_conceded', 'prior_last5_mean_bowling_runs_conceded', 'prior_last5_count_bowling_runs_conceded', 'prior_career_bowling_legal_balls', 'prior_last5_mean_bowling_legal_balls', 'prior_last5_count_bowling_legal_balls', 'prior_career_bowling_dots', 'prior_last5_mean_bowling_dots', 'prior_

## 2. Construct one sample per player-target match

Each sample uses up to five immediately preceding rows for the same player, ordered by match date and match ID. This notebook requires a full five-match history so all sequence tensors have the same length. The target is the next-match outcome attached to the final historical row.

The rows are sorted within each player; the sequence is never allowed to cross between players.

In [3]:
SEQUENCE_LENGTH = 5

def build_sequences(df, features, target, dataset_name):
    usable = df.loc[
        df["player_name"].notna()
        & df["date"].notna()
        & df["next_match_date"].notna()
        & df[target].notna()
    ].copy()
    usable[target] = pd.to_numeric(usable[target], errors="coerce")
    usable = usable.loc[usable[target].notna()].copy()
    for col in features:
        usable[col] = pd.to_numeric(usable[col], errors="coerce")
    usable = usable.sort_values(
        ["player_name", "date", "match_id"], kind="stable"
    ).reset_index(drop=True)

    X_list, y_list, metadata = [], [], []
    for player_name, group in usable.groupby("player_name", sort=False, dropna=True):
        group = group.reset_index(drop=True)
        for end_idx in range(SEQUENCE_LENGTH - 1, len(group)):
            window = group.iloc[end_idx - SEQUENCE_LENGTH + 1:end_idx + 1]
            target_row = group.iloc[end_idx]
            # If historical feature columns contain missing values, preserve them here;
            # imputation/scaling must be fitted only on the training partition in Day 12.
            X_list.append(window[features].to_numpy(dtype=np.float32))
            y_list.append(float(target_row[target]))
            metadata.append({
                "dataset": dataset_name,
                "target": target,
                "player_name": player_name,
                "history_start_match_id": str(window.iloc[0]["match_id"]),
                "history_end_match_id": str(window.iloc[-1]["match_id"]),
                "history_end_date": str(window.iloc[-1]["date"].date()),
                "target_match_date": str(target_row["next_match_date"].date()),
                "target_history_row_index": int(end_idx),
                "history_matches": SEQUENCE_LENGTH
            })

    if not X_list:
        return np.empty((0, SEQUENCE_LENGTH, len(features)), dtype=np.float32), np.empty((0,), dtype=np.float32), pd.DataFrame(metadata)
    return np.stack(X_list), np.asarray(y_list, dtype=np.float32), pd.DataFrame(metadata)

X_bat, y_bat, meta_bat = build_sequences(
    batting, bat_features, "next_match_runs", "batting"
)
X_w_wkts, y_w_wkts, meta_w_wkts = build_sequences(
    bowling, bowl_features, "next_match_wickets", "bowling"
)
X_w_runs, y_w_runs, meta_w_runs = build_sequences(
    bowling, bowl_features, "next_match_runs_conceded", "bowling"
)

print("Batting tensor:", X_bat.shape, "target:", y_bat.shape)
print("Bowling wickets tensor:", X_w_wkts.shape, "target:", y_w_wkts.shape)
print("Bowling runs-conceded tensor:", X_w_runs.shape, "target:", y_w_runs.shape)

Batting tensor: (15821, 5, 15) target: (15821,)
Bowling wickets tensor: (12335, 5, 15) target: (12335,)
Bowling runs-conceded tensor: (12335, 5, 15) target: (12335,)


## 3. Chronological train/validation/test partitions

We split sequence samples by **target match date** (70% / 15% / 15%), not by randomly shuffling them. This is still a row/sample-level split; multiple players from the same target match may fall on a boundary. The report records date boundaries so this limitation can be reviewed before model training.

In [4]:
def chronological_sequence_split(X, y, meta, dataset_name, target):
    if len(meta) != len(y) or len(X) != len(y):
        raise ValueError(f"{dataset_name}/{target}: X, y and metadata lengths differ.")
    if len(y) < 20:
        raise ValueError(
            f"{dataset_name}/{target}: only {len(y)} five-match sequences. "
            "Inspect coverage; do not silently train on an inadequate sample."
        )
    meta = meta.copy()
    meta["target_match_date"] = pd.to_datetime(meta["target_match_date"], errors="coerce")
    order = np.argsort(meta["target_match_date"].to_numpy(), kind="stable")
    X, y, meta = X[order], y[order], meta.iloc[order].reset_index(drop=True)
    n = len(y)
    train_end, val_end = int(n * 0.70), int(n * 0.85)
    if train_end < 2 or val_end <= train_end or val_end >= n:
        raise ValueError(f"{dataset_name}/{target}: cannot create non-empty partitions.")
    slices = {
        "train": slice(0, train_end),
        "validation": slice(train_end, val_end),
        "test": slice(val_end, n),
    }
    split_rows = []
    for split_name, slc in slices.items():
        part_dates = meta.iloc[slc]["target_match_date"]
        split_rows.append({
            "dataset": dataset_name,
            "target": target,
            "split": split_name,
            "rows": len(meta.iloc[slc]),
            "target_date_start": str(part_dates.min().date()),
            "target_date_end": str(part_dates.max().date()),
        })
    return {
        name: (X[slc], y[slc], meta.iloc[slc].copy())
        for name, slc in slices.items()
    }, split_rows

sequence_tasks = [
    ("batting", "next_match_runs", X_bat, y_bat, meta_bat, bat_features),
    ("bowling", "next_match_wickets", X_w_wkts, y_w_wkts, meta_w_wkts, bowl_features),
    ("bowling", "next_match_runs_conceded", X_w_runs, y_w_runs, meta_w_runs, bowl_features),
]
splits = {}
split_report_rows = []
for dataset_name, target, X, y, meta, features in sequence_tasks:
    try:
        split_result, split_rows = chronological_sequence_split(
            X, y, meta, dataset_name, target
        )
        splits[(dataset_name, target)] = split_result
        split_report_rows.extend(split_rows)
        print(f"READY: {dataset_name}/{target} -> {len(y)} samples")
    except Exception as exc:
        print(f"NOT READY: {dataset_name}/{target}: {exc}")

split_report = pd.DataFrame(split_report_rows)
display(split_report)

READY: batting/next_match_runs -> 15821 samples
READY: bowling/next_match_wickets -> 12335 samples
READY: bowling/next_match_runs_conceded -> 12335 samples


,dataset,target,split,rows,target_date_start,target_date_end
0,batting,next_match_runs,train,11074,2008-05-03,2022-04-10
1,batting,next_match_runs,validation,2373,2022-04-10,2024-04-29
2,batting,next_match_runs,test,2374,2024-04-29,2026-05-31
3,bowling,next_match_wickets,train,8634,2008-05-03,2022-04-02
4,bowling,next_match_wickets,validation,1850,2022-04-02,2024-04-26
5,bowling,next_match_wickets,test,1851,2024-04-26,2026-05-31
6,bowling,next_match_runs_conceded,train,8634,2008-05-03,2022-04-02
7,bowling,next_match_runs_conceded,validation,1850,2022-04-02,2024-04-26
8,bowling,next_match_runs_conceded,test,1851,2024-04-26,2026-05-31


## 4. Leakage and consistency checks

These checks validate tensor shapes, target alignment metadata, player identity, chronological ordering, and feature naming. A `prior_` prefix is a convention, not proof of leakage safety; confirm the Day 4 construction logic for every feature before training.

In [5]:
check_rows = []
for dataset_name, target, X, y, meta, features in sequence_tasks:
    status, detail = "PASS", ""
    try:
        if X.ndim != 3:
            raise ValueError(f"Expected 3D tensor, found ndim={X.ndim}")
        if X.shape[1] != SEQUENCE_LENGTH:
            raise ValueError(f"Sequence length is {X.shape[1]}, expected {SEQUENCE_LENGTH}")
        if X.shape[2] != len(features):
            raise ValueError("Tensor feature dimension does not match feature manifest.")
        if len(X) != len(y) or len(y) != len(meta):
            raise ValueError("X/y/metadata sample counts differ.")
        if meta["player_name"].isna().any():
            raise ValueError("Missing player identity in sequence metadata.")
        if not meta["history_matches"].eq(SEQUENCE_LENGTH).all():
            raise ValueError("Unexpected sequence length in metadata.")
        if any(not c.startswith("prior_") for c in features):
            raise ValueError("A candidate feature does not start with prior_.")
        if not np.isfinite(y).all():
            raise ValueError("Target contains non-finite values.")
        if not np.isfinite(X).all():
            status = "REVIEW"
            detail = "Feature tensor contains NaN/inf; fit imputation on training data in Day 12."
    except Exception as exc:
        status, detail = "FAIL", repr(exc)
    check_rows.append({
        "dataset": dataset_name, "target": target,
        "samples": len(y), "sequence_length": SEQUENCE_LENGTH,
        "feature_count": len(features), "status": status, "detail": detail
    })

checks_df = pd.DataFrame(check_rows)
display(checks_df)

,dataset,target,samples,sequence_length,feature_count,status,detail
0,batting,next_match_runs,15821,5,15,REVIEW,Feature tensor contains NaN/inf; fit imputatio...
1,bowling,next_match_wickets,12335,5,15,REVIEW,Feature tensor contains NaN/inf; fit imputatio...
2,bowling,next_match_runs_conceded,12335,5,15,REVIEW,Feature tensor contains NaN/inf; fit imputatio...


## 5. Save tensors, metadata, feature manifests, and documentation

Each target gets separate `.npz` tensors and metadata. The manifest preserves the exact feature order. No imputer or scaler is fitted today; Day 12 must fit preprocessing using training data only.

In [6]:
sequence_manifest_rows = []
saved_rows = []

for dataset_name, target, X, y, meta, features in sequence_tasks:
    safe_target = target.replace("/", "_")
    prefix = f"{dataset_name}_{safe_target}"
    npz_path = DAY11_DIR / f"{prefix}_sequences.npz"
    metadata_path = DAY11_DIR / f"{prefix}_metadata.csv"
    feature_path = DAY11_DIR / f"{prefix}_features.json"

    np.savez_compressed(npz_path, X=X, y=y)
    meta.to_csv(metadata_path, index=False)
    feature_payload = {
        "dataset": dataset_name,
        "target": target,
        "sequence_length": SEQUENCE_LENGTH,
        "feature_columns_in_order": features,
        "tensor_shape": list(X.shape),
        "sample_count": int(len(y)),
        "target_definition": "Next-match target stored on the last historical row in each five-match window."
    }
    feature_path.write_text(json.dumps(feature_payload, indent=2), encoding="utf-8")
    sequence_manifest_rows.append({
        "dataset": dataset_name,
        "target": target,
        "sequence_length": SEQUENCE_LENGTH,
        "feature_count": len(features),
        "sample_count": len(y),
        "tensor_shape_json": json.dumps(list(X.shape)),
        "tensor_path": str(npz_path),
        "metadata_path": str(metadata_path),
        "feature_manifest_path": str(feature_path)
    })
    saved_rows.append({
        "dataset": dataset_name, "target": target,
        "tensor_file": str(npz_path), "metadata_file": str(metadata_path),
        "feature_manifest_file": str(feature_path)
    })

sequence_manifest = pd.DataFrame(sequence_manifest_rows)
sequence_manifest.to_csv(REPORT_DIR / "sequence_manifest.csv", index=False)
split_report.to_csv(REPORT_DIR / "chronological_split_summary.csv", index=False)
checks_df.to_csv(REPORT_DIR / "sequence_validation_checks.csv", index=False)

print("Saved sequence outputs:")
display(pd.DataFrame(saved_rows))

Saved sequence outputs:


,dataset,target,tensor_file,metadata_file,feature_manifest_file
0,batting,next_match_runs,D:\Git\Personal\ipl-player-performance\data\pr...,D:\Git\Personal\ipl-player-performance\data\pr...,D:\Git\Personal\ipl-player-performance\data\pr...
1,bowling,next_match_wickets,D:\Git\Personal\ipl-player-performance\data\pr...,D:\Git\Personal\ipl-player-performance\data\pr...,D:\Git\Personal\ipl-player-performance\data\pr...
2,bowling,next_match_runs_conceded,D:\Git\Personal\ipl-player-performance\data\pr...,D:\Git\Personal\ipl-player-performance\data\pr...,D:\Git\Personal\ipl-player-performance\data\pr...


## 6. Day 11 acceptance checklist

- [ ] All three targets have sequence tensors, metadata, and feature manifests.
- [ ] Each tensor shape is `(number_of_samples, 5, number_of_features)`.
- [ ] No sample has fewer than five historical matches.
- [ ] Chronological split reports exist for each target.
- [ ] Validation checks contain no `FAIL` status. Investigate `REVIEW` before modelling.
- [ ] Feature definitions are checked against Day 4 to confirm no current-match or future information leaks into `prior_` columns.
- [ ] `docs/day11_sequence_dataset_preparation.md` is generated from the actual run.

Do not begin LSTM/GRU training until the actual status table and reports are reviewed.

In [7]:
def frame_text(frame):
    return frame.to_string(index=False) if frame is not None and not frame.empty else "No rows produced."

doc = f"""# Day 11 — Sequence Dataset Preparation

## Execution summary
### Validation checks
```text
{frame_text(checks_df)}
```

### Chronological splits
```text
{frame_text(split_report)}
```

### Sequence manifest
```text
{frame_text(sequence_manifest)}
```

## Method
- Sequence length: {SEQUENCE_LENGTH} historical matches per sample.
- Input features: numeric `prior_...` columns only, saved in a per-target feature manifest.
- Target values: `next_match_runs`, `next_match_wickets`, and `next_match_runs_conceded`.
- Samples are ordered by player and historical match date; sequences do not cross player boundaries.
- The final historical row stores the next-match target.
- Train/validation/test partitions are ordered by target match date at 70% / 15% / 15%.
- No imputer or scaler is fitted during this dataset-preparation day.

## Outputs
- `data/processed/day11_sequences/*.npz`
- `data/processed/day11_sequences/*_metadata.csv`
- `data/processed/day11_sequences/*_features.json`
- `reports/day11/sequence_manifest.csv`
- `reports/day11/chronological_split_summary.csv`
- `reports/day11/sequence_validation_checks.csv`

## Limitations and review notes
- The `prior_` prefix is a screening convention, not proof that a feature is leakage-free.
- This is a sample-level chronological split; samples for different players in the same match can straddle boundaries.
- Missing input features are preserved as NaN so Day 12 can fit imputation only on training data.
- Sequence construction is based on rows present in the Day 4 player-match feature tables; verify that rows represent the intended player-match chronology and that next-match targets align correctly.

## Next
Day 12: train LSTM and GRU regressors with training-only preprocessing, early stopping where supported, and regression metrics.
"""
doc_path = DOCS_DIR / "day11_sequence_dataset_preparation.md"
doc_path.write_text(doc, encoding="utf-8")
print("Saved documentation:", doc_path)
print("Checks:")
display(checks_df)

Saved documentation: D:\Git\Personal\ipl-player-performance\docs\day11_sequence_dataset_preparation.md
Checks:


,dataset,target,samples,sequence_length,feature_count,status,detail
0,batting,next_match_runs,15821,5,15,REVIEW,Feature tensor contains NaN/inf; fit imputatio...
1,bowling,next_match_wickets,12335,5,15,REVIEW,Feature tensor contains NaN/inf; fit imputatio...
2,bowling,next_match_runs_conceded,12335,5,15,REVIEW,Feature tensor contains NaN/inf; fit imputatio...
